In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
import os
from google.colab import drive
drive.mount('/content/drive')

pd.set_option('display.float_format', '{:,.0f}'.format)

Mounted at /content/drive


In [ ]:
from sklearn.model_selection import StratifiedKFold

In [ ]:
%cd "drive/MyDrive/Colab Notebooks/MIS 769/AnalyzeMe1"

/content/drive/MyDrive/Colab Notebooks/MIS 769/AnalyzeMe1


In [ ]:
# Adjust paths if the files are not in the current directory
df_train = pd.read_csv('train_final_ready.csv')
df_test  = pd.read_csv('test_final_ready.csv')

print("Train shape:", df_train.shape)
print("Test shape: ", df_test.shape)

Train shape: (590540, 31)
Test shape:  (506691, 30)


In [ ]:
cat_cols = [
    'ProductCD', 'card4', 'card6', 'P_email_grouped_v2',
    'card3_bin', 'card5_bin',
    'id_12', 'id_15', 'id_28', 'id_29', 'id_35', 'id_36', 'id_37', 'id_38',
    'DeviceType', 'DeviceInfo_grouped_adv', 'id_01_risk_bin'
]

for col in cat_cols:
    if col in df_train.columns:
        df_train[col] = df_train[col].astype('category')
    if col in df_test.columns:
        df_test[col] = df_test[col].astype('category')

In [ ]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 506691 entries, 0 to 506690
Data columns (total 30 columns):
 #   Column                  Non-Null Count   Dtype   
---  ------                  --------------   -----   
 0   TransactionID           506691 non-null  int64   
 1   TransactionAmt          506691 non-null  float64 
 2   ProductCD               506691 non-null  category
 3   card4                   503605 non-null  category
 4   card6                   503684 non-null  category
 5   id_02                   136976 non-null  float64 
 6   id_12                   141907 non-null  category
 7   id_15                   136977 non-null  category
 8   id_28                   136778 non-null  category
 9   id_29                   136778 non-null  category
 10  id_35                   136977 non-null  category
 11  id_36                   136977 non-null  category
 12  id_37                   136977 non-null  category
 13  id_38                   136977 non-null  category
 14  Devi

In [ ]:
# Standardize all column names to lowercase (or uppercase — pick one)
df_train.columns = df_train.columns.str.lower()
df_test.columns  = df_test.columns.str.lower()

# Now the M columns should match: m1_checked, m6_checked, etc.

In [ ]:
print("Columns only in train after lowercasing:", set(df_train.columns) - set(df_test.columns))
print("Columns only in test after lowercasing: ", set(df_test.columns) - set(df_train.columns))
print("Common columns count:", len(set(df_train.columns) & set(df_test.columns)))

Columns only in train after lowercasing: {'isfraud'}
Columns only in test after lowercasing:  set()
Common columns count: 30


In [ ]:
# Exclude ID and target (now lowercase)
exclude = ['transactionid', 'isfraud']

# Features = columns present in both (minus exclude)
features = [col for col in df_train.columns if col not in exclude]

# Confirm we're good
print(f"Number of features: {len(features)}")
print("Features:", features)  # optional — just to visually check

X      = df_train[features]
y      = df_train['isfraud'].astype(int)   # ensure it's 0/1 int
X_test = df_test[features]

Number of features: 29
Features: ['transactionamt', 'productcd', 'card4', 'card6', 'has_dist2', 'has_dist1', 'm1_checked', 'm6_checked', 'm8_checked', 'm4_checked', 'addr1_present', 'r_email_present', 'p_email_grouped_v2', 'card3_bin', 'card5_bin', 'hour', 'id_02', 'id_12', 'id_15', 'id_28', 'id_29', 'id_35', 'id_36', 'id_37', 'id_38', 'devicetype', 'deviceinfo_grouped_adv', 'id_01_risk_bin', 'id_11_high']


In [ ]:
cat_features = []
for i, col in enumerate(features):
    if pd.api.types.is_categorical_dtype(X[col]):
        cat_features.append(i)

print(f"Number of categorical features found: {len(cat_features)}")
print("Indices:", cat_features)

Number of categorical features found: 17
Indices: [1, 2, 3, 12, 13, 14, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27]


/tmp/ipython-input-266/1880730231.py:3: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(X[col]):


In [ ]:
print("Checking categorical columns:")
for col in cat_cols:
    if col in X.columns:
        print(f"{col:25} dtype: {X[col].dtype}   | in test? {col in X_test.columns}")
    else:
        print(f"{col:25} MISSING in features")

Checking categorical columns:
productcd                 dtype: category   | in test? True
card4                     dtype: category   | in test? True
card6                     dtype: category   | in test? True
p_email_grouped_v2        dtype: category   | in test? True
card3_bin                 dtype: category   | in test? True
card5_bin                 dtype: category   | in test? True
id_12                     dtype: category   | in test? True
id_15                     dtype: category   | in test? True
id_28                     dtype: category   | in test? True
id_29                     dtype: category   | in test? True
id_35                     dtype: category   | in test? True
id_36                     dtype: category   | in test? True
id_37                     dtype: category   | in test? True
id_38                     dtype: category   | in test? True
devicetype                dtype: category   | in test? True
deviceinfo_grouped_adv    dtype: category   | in test? True
id_01_risk

In [ ]:
cat_features = [features.index(col) for col in cat_cols if col in features]
print(f"Number of categorical features: {len(cat_features)}")

Number of categorical features: 13


In [ ]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Data columns (total 29 columns):
 #   Column                  Non-Null Count   Dtype   
---  ------                  --------------   -----   
 0   transactionamt          590540 non-null  float64 
 1   productcd               590540 non-null  category
 2   card4                   588963 non-null  category
 3   card6                   588969 non-null  category
 4   has_dist2               590540 non-null  int64   
 5   has_dist1               590540 non-null  int64   
 6   m1_checked              590540 non-null  int64   
 7   m6_checked              590540 non-null  int64   
 8   m8_checked              590540 non-null  int64   
 9   m4_checked              590540 non-null  int64   
 10  addr1_present           590540 non-null  int64   
 11  r_email_present         590540 non-null  int64   
 12  p_email_grouped_v2      590540 non-null  category
 13  card3_bin               588975 non-null  category
 14  card

In [ ]:
# LightGBM parameters (solid baseline for fraud data)
params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.02,
    'num_leaves': 180,
    'max_depth': 9,
    'min_data_in_leaf': 150,
    'feature_fraction': 0.6,
    'bagging_fraction': 0.9,
    'bagging_freq': 5,
    'lambda_l1': 0.5,
    'lambda_l2': 1.0,
    'is_unbalance': True,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

# Stratified 5-fold CV (simple, balanced, good starting point)
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"\nFold {fold+1}/5")

    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    # Create Datasets with categorical indices
    train_data = lgb.Dataset(
        X_tr,
        label=y_tr,
        categorical_feature=cat_features,   # ← your list of 17 indices
        free_raw_data=False
    )
    val_data = lgb.Dataset(
        X_val,
        label=y_val,
        categorical_feature=cat_features,
        reference=train_data,
        free_raw_data=False
    )

    # Train with early stopping
    model = lgb.train(
        params,
        train_data,
        num_boost_round=15000,
        valid_sets=[train_data, val_data],
        valid_names=['train', 'valid'],
        callbacks=[
            lgb.early_stopping(stopping_rounds=600, verbose=True),
            lgb.log_evaluation(period=200)
        ]
    )

    # Predict on validation and test
    oof_preds[val_idx] = model.predict(X_val)
    test_preds += model.predict(X_test) / kf.n_splits

    # Fold AUC
    fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
    print(f"Fold AUC: {fold_auc:.5f}")

# Final OOF AUC (your main local score)
full_oof_auc = roc_auc_score(y, oof_preds)
print(f"\nFull OOF CV AUC: {full_oof_auc:.5f}")




Fold 1/5
Training until validation scores don't improve for 600 rounds
[200]	train's auc: 0.85844	valid's auc: 0.836628
[400]	train's auc: 0.874671	valid's auc: 0.842792
[600]	train's auc: 0.885526	valid's auc: 0.845852
[800]	train's auc: 0.894728	valid's auc: 0.847797
[1000]	train's auc: 0.901986	valid's auc: 0.848994
[1200]	train's auc: 0.907348	valid's auc: 0.849477
[1400]	train's auc: 0.912504	valid's auc: 0.849703
[1600]	train's auc: 0.917276	valid's auc: 0.849998
[1800]	train's auc: 0.921764	valid's auc: 0.849999
[2000]	train's auc: 0.925343	valid's auc: 0.84988
[2200]	train's auc: 0.928923	valid's auc: 0.849663
Early stopping, best iteration is:
[1725]	train's auc: 0.920149	valid's auc: 0.850201
Fold AUC: 0.85020

Fold 2/5
Training until validation scores don't improve for 600 rounds
[200]	train's auc: 0.858587	valid's auc: 0.837252
[400]	train's auc: 0.874829	valid's auc: 0.844585
[600]	train's auc: 0.885576	valid's auc: 0.848388
[800]	train's auc: 0.893592	valid's auc: 0.8508

In [ ]:
importance = model.feature_importance(importance_type='gain')  # or 'split'
feat_imp = pd.DataFrame({'feature': features, 'importance': importance})
feat_imp = feat_imp.sort_values('importance', ascending=False).head(15)
print(feat_imp)

                   feature  importance
0           transactionamt   3,140,033
15                    hour   1,352,550
11         r_email_present   1,175,843
16                   id_02   1,007,104
1                productcd     947,495
12      p_email_grouped_v2     760,064
3                    card6     562,390
14               card5_bin     525,011
13               card3_bin     466,954
26  deviceinfo_grouped_adv     369,765
2                    card4     285,374
25              devicetype     149,310
21                   id_35     137,189
20                   id_29     128,370
9               m4_checked     126,078


In [ ]:
df_train['oof_pred'] = oof_preds
df_train.to_csv('train_with_oof.csv', index=False)